# A1 — Equity Curves per Window

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    rk = wm['result_key']
    trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
    trades = trades.sort_values('entry_time').reset_index(drop=True)
    trades['cum_gross'] = trades['gross_usd'].cumsum()
    ax.step(range(len(trades)), trades['cum_gross'], where='post',
            color=WIN_COLORS[wk], linewidth=1.5, label='Baseline')
    if 'fomc_date' in wm:
        # mark approximate fomc trade index
        fd = wm['fomc_date']
        fomc_mask = trades['entry_time'].dt.date.astype(str) == fd
        if fomc_mask.any():
            ax.axvline(fomc_mask.idxmax(), color='orange', ls='--', alpha=0.7, label='FOMC')
    ax.axhline(0, color='black', lw=0.5)
    bs = BASELINE_STATS[wk]
    ax.set_title(f"{wk}: {wm['front']}→{wm['back']}  n={bs['n']}  WR={bs['wr']:.1%}  Gross=${bs['gross']:.2f}")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative Gross P&L ($)')
    ax.legend(fontsize=8)
fig.suptitle('Equity Curves — All 4 Roll Windows (Baseline)', fontsize=13)
save_fig(fig, 'A1_equity_curves_per_window.png')
